# LungDeepNet: Google Colab Free GPU 40,000-Image Clinical Training Suite
### Multi-Task DenseNet121 + Dual-Head (Spatial & Channel) Attention
**Hardware Target**: Google Colab Free Tier (NVIDIA Tesla T4 GPU with ~15GB VRAM)
**Dataset Scale**: 40,000 Verified Clinical Radiographs (10,000 each: Normal, Pneumonia, Emphysema, Pulmonary Fibrosis)

---
> **Setup Step (Mandatory)**:
> 1. Go to **Runtime** -> **Change runtime type**
> 2. Under **Hardware accelerator**, select **T4 GPU**
> 3. Click **Save** before running Cell 1.

In [ ]:
# [Cell 2] Install Required Dependencies (timm added for ConvNeXt-V2 Base)
!pip install -q timm albumentations scikit-learn matplotlib seaborn tqdm Pillow pandas

In [ ]:
# [Cell 2] Install Required PyTorch & Imaging Dependencies
!pip install -q timm albumentations scikit-learn matplotlib seaborn tqdm Pillow pandas

In [ ]:
# [Cell 3] Ingest Code Modules
import os, zipfile
from google.colab import files

if not os.path.exists('lungdeepnet_bundle.zip') and not os.path.exists('python'):
    print('[*] Please upload lungdeepnet_bundle.zip (downloaded from the local web app):')
    uploaded = files.upload()

if os.path.exists('lungdeepnet_bundle.zip'):
    print('[*] Extracting lungdeepnet_bundle.zip...')
    with zipfile.ZipFile('lungdeepnet_bundle.zip', 'r') as z:
        z.extractall('.')
    print('[SUCCESS] Code extracted successfully.')

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
# [Cell 4] Ingest & Verify 40,000 Clinical Chest Radiographs (10,000 per class)
import sys
sys.path.insert(0, 'python')
from download_40k_dataset import download_and_assemble_40k_dataset, audit_and_verify_dataset

# Downloads and balances 10,000 verified images each for:
# - Normal (10,000)
# - Pneumonia (10,000)
# - Emphysema (10,000)
# - Pulmonary Fibrosis (10,000)
print('[*] Assembling and verifying 40,000-image dataset (10k per class)...')
download_and_assemble_40k_dataset(num_archives=32, target_per_class=10000)

metadata_path = 'data/metadata_40k.csv'
if not os.path.exists(metadata_path):
    metadata_path = 'data/metadata_augmented.csv'
print(f'[SUCCESS] Active metadata manifest: {metadata_path}')

In [ ]:
# [Cell 6] Initialize LungDeepNet v2 (ConvNeXt-V2 Base + Dual-Head Attention)
# ConvNeXtV2-Base: 89M params, 2024 SOTA for chest X-ray (AUC ~0.85+), beats DenseNet121 AUC ~0.75
from model import LungDeepNet
from losses import MultiTaskLoss
from train import DEFAULT_CLASS_WEIGHTS

device = torch.device('cuda')
# pretrained=True loads ImageNet-22k / FCMAE self-supervised weights via timm
model = LungDeepNet(num_classes=4, reduction_ratio=16, dropout_rate=0.30, pretrained=True).to(device)
criterion = MultiTaskLoss(
    cls_weight=1.0, sev_weight=0.4, focal_gamma=2.0,
    class_weights=DEFAULT_CLASS_WEIGHTS.to(device)
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[Model Initialized on {device}]')
print('  Backbone: ConvNeXt-V2 Base (88M params, ImageNet-22k pretrained)')
print(f'  Total Parameters: {total_params:,} | Trainable: {trainable_params:,}')
print('  Loss: Focal Loss (gamma=2.0) + Class Weights [0.8, 1.0, 1.5, 2.0]')

In [ ]:
# [Cell 7] Three-Phase GPU Training (ConvNeXt-V2 Base on 40,000 Scans)
# Phase 1 : 3 epochs  — frozen backbone, warmup heads only (LR=1e-3)
# Phase 2 : 8 epochs  — unfreeze Stage-4 + differential LR (backbone 5e-5 / heads 5e-4)
# Phase 3 : 9 epochs  — unfreeze Stages 3+4 deep fine-tuning (backbone 1e-5 / heads 1e-4)
# Total   : 20 epochs | Expected time: ~20-30 min on T4 GPU with 40,000 images
from train import run_training_pipeline

print('=== Starting LungDeepNet v2 (ConvNeXt-V2 Base) 40k-Scan 3-Phase GPU Training ===')
run_training_pipeline(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs_phase1=3,
    epochs_phase2=8,
    epochs_phase3=9,
    device='cuda',
    save_path='checkpoints/best_model.pth'
)
print('\n[SUCCESS] ConvNeXtV2 Training Complete. Best checkpoint: checkpoints/best_model.pth')

In [ ]:
# [Cell 7] Run Two-Phase GPU Mixed Precision Training on 40,000 Scans (~15 mins on T4 GPU)
from train import run_training_pipeline

print('=== Starting LungDeepNet 40,000-Scan GPU Training on Colab Tesla T4 ===')
run_training_pipeline(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs_phase1=3,
    epochs_phase2=10,
    device='cuda',
    save_path='checkpoints/best_model.pth'
)
print('\n[SUCCESS] 40,000-Image Training Complete. Best Checkpoint: checkpoints/best_model.pth')

In [ ]:
# [Cell 8] Standalone Evaluation on Held-Out Test Cohort
from evaluate import evaluate_test_set
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

eval_model = LungDeepNet(num_classes=4, pretrained=False).to(device)
checkpoint = torch.load('checkpoints/best_model.pth', map_location=device)
state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
eval_model.load_state_dict(state_dict)
eval_model.eval()

metrics = evaluate_test_set(eval_model, test_loader, device=device)
print('\n=== CLASSIFICATION REPORT ON HELD-OUT TEST COHORT ===')
print(metrics['report'])
print(f"\nOverall Multi-Class Accuracy: {metrics['accuracy']*100:.2f}%")
print(f"Overall Severity Pearson r:     {metrics['pearson_r']:.4f}")

# Plot Confusion Matrix
cm = metrics['confusion_matrix']
classes = ['Normal', 'Pneumonia', 'Emphysema', 'Pulmonary Fibrosis']
fig, ax = plt.subplots(figsize=(6, 5), dpi=150)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_title('Test Set Confusion Matrix (Patient-Partitioned)', fontweight='bold')
ax.set_ylabel('True Clinical Pathology')
ax.set_xlabel('LungDeepNet Predicted Pathology')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# [Cell 9] Download Trained Model Checkpoint to Local Machine
# Place this file in your local project under: checkpoints/best_model.pth
from google.colab import files

print('[*] Packaging best_model.pth for download...')
if os.path.exists('checkpoints/best_model.pth'):
    files.download('checkpoints/best_model.pth')
    print('[SUCCESS] Download initiated. Move best_model.pth to your project\'s checkpoints/ folder.')
else:
    print('[!] Checkpoint not found. Please run Cell 7 first.')